# Lesson 04 Lab — A Tail-Safe Vector Kernel

**Puzzle:** When program_id, tl.arange, mask, and BLOCK change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates program_id, tl.arange, mask, and BLOCK and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

A vector kernel derives offsets from program_id and a compile-time BLOCK. The runtime element count controls a tail mask, while BLOCK shapes the compiler-visible tensor. Correctness at non-divisible lengths is part of the kernel contract.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["program_id, tl.arange, mask, and BLOCK"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: named PyTorch CUDA/library or standard-grid path. Candidate: reviewed Triton kernel or explicit model described below.

Omitting the load and store mask may appear correct on aligned examples while reading or writing beyond the allocation on the first awkward shape.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 4
LESSON_TITLE = 'A Tail-Safe Vector Kernel'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260817
}


## 5. Freeze the experiment

**Experiment:** Sweep four BLOCK values on an odd vector length and verify every tail element.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": 512,
  "secondary": 0.018719999119639397,
  "max_abs_error": 4.76837158203125e-07,
  "passed": true,
  "details": {
    "n_elements": 4194321,
    "tail": 17,
    "block_sweep": {
      "128": {
        "median_ms": 0.028672000393271446,
        "samples_ms": [
          0.036959998309612274,
          0.0331839993596077,
          0.031168000772595406,
          0.03017600066959858,
          0.028063999488949776,
          0.028384000062942505,
          0.029440000653266907,
          0.028672000393271446,
          0.029311999678611755,
          0.028224000707268715,
          0.027648000046610832,
          0.027488000690937042,
          0.028831999748945236,
          0.02703999914228916,
          0.026335999369621277
        ]
      },
      "256": {
        "median_ms": 0.019648000597953796,
        "samples_ms": [
          0.027456000447273254,
          0.025439999997615814,
          0.0208320003002882,
          0.02099199965596199,
          0.0213120002

## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Best BLOCK | 512 |
| Best median | 0.0187 ms |
| Maximum absolute error | 4.768e-07 |
| Acceptance gate | true |


## 8. Explain without overclaiming

The odd-length tail remained correct. BLOCK=512 was fastest in this four-point sweep at 0.0187 ms.

A named Triton or PyTorch CUDA path executed on the recorded GPU. The result applies to the printed shape, dtype, implementation, and software stack; internal hardware causes require profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'native-backend',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Choose BLOCK from measured candidates, but never trade away the runtime tail mask for a prettier grid.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 4,
  "title": "A Tail-Safe Vector Kernel",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260817
  },
  "evidence_label": "native-backend",
  "metrics": {
    "primary": 512,
    "secondary": 0.018719999119639397,
    "max_abs_error": 4.76837158203125e-07,
    "passed": true,
    "details": {
      "n_elements": 4194321,
      "tail": 17,
      "block_sweep": {
        "128": {
          "median_ms": 0.028672000393271446,
          "samples_ms": [
            0.036959998309612274,
            0.0331839993596077,
            0.031168000772595406,
            0.03017600066959858,
            0.028063999488949776,
            0.028384000062942505,
            0.029440000653266907,
            0.028672000393271446,
            0.029311999678611

## 10. Make the bounded decision

> Choose BLOCK from measured candidates, but never trade away the runtime tail mask for a prettier grid.

**Failure analysis:** Omitting the load and store mask may appear correct on aligned examples while reading or writing beyond the allocation on the first awkward shape.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
